# NB18b — School Finance Net Patch (WDE FY2025 Model)

Patches the `school_finance_net` field in `data/processed/wy_fiscal_coefficients.json` (49 actions × 23 counties).

The prior values used a 37-mill proxy (constant \$6,084,650 for all counties, `school_finance_net_total = null`).
This notebook replaces those with real WDE FY2025 model data:

- **Source 1**: Wyoming Block Grant Estimated Payment Details — FY2025 Wyoming Funding Model Version 3(d)
- **Source 2**: Wyoming School District Locally Collected Revenues — FY2024 actuals / PY2024 mineral estimates

Key steps:
1. Parse entitlement/recapture from Payment Details CSV → `net_flow = entitlement - recapture` per district
2. Parse 6-mill (82110/82111) split and mineral revenue from LCR CSV
3. Aggregate district → county with cross-county apportionment (4 clusters)
4. Zero-leakage assertion
5. Patch coefficients JSON + append source rows


In [ ]:
import csv
import json
import re
from pathlib import Path
from collections import defaultdict

# ── Path constants ─────────────────────────────────────────────────────────────
ROOT = Path("/Users/dylanhartman/Library/CloudStorage/OneDrive-UniversityofWyoming/Research/Energy Modeling/energy-map")
RAW_DIR  = ROOT / "data" / "raw" / "wy_fiscal" / "extracted"

PAYMENT_CSV = RAW_DIR / " Wyoming Block Grant Estimated Payment Details  - Sheet1.csv"
LCR_CSV     = RAW_DIR / "Wyoming School District Locally Collected Revenues  - Sheet1.csv"

COEFF_FILE   = ROOT / "data" / "processed" / "wy_fiscal_coefficients.json"
SOURCES_FILE = ROOT / "data" / "processed" / "wy_fiscal_sources.csv"

# ── GEOID mappings ─────────────────────────────────────────────────────────────
DISTRICT_PREFIX_TO_GEOID = {
    "01": "56001", "02": "56003", "03": "56005", "04": "56007",
    "05": "56009", "06": "56011", "07": "56013", "08": "56015",
    "09": "56017", "10": "56019", "11": "56021", "12": "56023",
    "13": "56025", "14": "56027", "15": "56029", "16": "56031",
    "17": "56033", "18": "56035", "19": "56037", "20": "56039",
    "21": "56041", "22": "56043", "23": "56045",
}

GEOID_TO_COUNTY = {
    "56001": "Albany",     "56003": "Big Horn",  "56005": "Campbell",
    "56007": "Carbon",     "56009": "Converse",  "56011": "Crook",
    "56013": "Fremont",    "56015": "Goshen",    "56017": "Hot Springs",
    "56019": "Johnson",    "56021": "Laramie",   "56023": "Lincoln",
    "56025": "Natrona",    "56027": "Niobrara",  "56029": "Park",
    "56031": "Platte",     "56033": "Sheridan",  "56035": "Sublette",
    "56037": "Sweetwater", "56039": "Teton",     "56041": "Uinta",
    "56043": "Washakie",   "56045": "Weston",
}

# ── Cross-county clusters ───────────────────────────────────────────────────────
# For 2-county clusters: cross_dest is a string GEOID
# For 3-county cluster: cross_dest is a list of 2 GEOIDs
# Park (prefix 15) has no cross-county assignment (82111 goes back to Park)
CROSS_COUNTY_CLUSTERS = {
    "big_horn_park": {
        "02": ("56003", "56029"),   # Big Horn → cross from Park
        "15": ("56029", None),      # Park → no cross assignment
    },
    "lincoln_sublette": {
        "12": ("56023", "56035"),   # Lincoln → cross from Sublette
        "18": ("56035", "56023"),   # Sublette → cross from Lincoln
    },
    "goshen_niobrara_platte": {
        "08": ("56015", ["56027", "56031"]),
        "14": ("56027", ["56015", "56031"]),
        "16": ("56031", ["56015", "56027"]),
    },
    "carbon_sweetwater": {
        "04": ("56007", "56037"),
        "19": ("56037", "56007"),
    },
}

print("Paths and constants loaded.")
print(f"  PAYMENT_CSV : {PAYMENT_CSV}")
print(f"  LCR_CSV     : {LCR_CSV}")
print(f"  COEFF_FILE  : {COEFF_FILE}")
print(f"  SOURCES_FILE: {SOURCES_FILE}")
print(f"  Counties mapped: {len(DISTRICT_PREFIX_TO_GEOID)}")


In [ ]:
# ── Dollar parsing ─────────────────────────────────────────────────────────────
def parse_dollar(s):
    """Parse WDE-formatted dollar strings to float.

    Examples:
        ' $ 49,686,768.46 '  →  49686768.46
        ' $ -   '            →  0.0
        '$(1,234.56)'        →  -1234.56
        '$0.00'              →  0.0
        ''                   →  0.0
    """
    s = s.strip()
    if not s:
        return 0.0
    # Parentheses-negative: $(1,234.56)
    paren_match = re.match(r'^\$?\s*\(([\d,\.]+)\)$', s)
    if paren_match:
        return -float(paren_match.group(1).replace(',', ''))
    # Remove $ and commas
    s = s.replace('$', '').replace(',', '').strip()
    # Handle '$ -' or '-' or '-   ' patterns → 0
    if re.match(r'^-\s*$', s):
        return 0.0
    return float(s)


# ── Parse Payment Details CSV ──────────────────────────────────────────────────
district_flows = {}  # district_id → dict

with open(PAYMENT_CSV, newline='', encoding='utf-8-sig') as f:
    all_rows = list(csv.reader(f))

# Find header row (contains 'District ID')
header_row_idx = None
for i, row in enumerate(all_rows):
    if row and row[0].strip() == 'District ID':
        header_row_idx = i
        break

assert header_row_idx is not None, "Could not find header row in Payment Details CSV"
print(f"Header row index: {header_row_idx}")
print(f"Header: {all_rows[header_row_idx]}")

data_rows = all_rows[header_row_idx + 1:]

for row in data_rows:
    if not row or not row[0].strip():
        continue  # skip blank rows
    raw_id = row[0].strip()
    if raw_id.startswith('77'):
        continue  # skip statewide total

    # Normalize to 7 chars with leading zero if needed
    district_id = raw_id.zfill(7)
    name = row[1].strip()

    # Col indices (0-based):
    # 0: District ID, 1: Name
    # 2: Model Generated Resources, 3: Total Reimbursements
    # 4: Foundation Program Block Grant Guarantee
    # 5: Total School District Local Revenues
    # 6: Estimated School District Entitlement Amount
    # 7: Entitlement % of Guarantee
    # 8: Estimated School District Recapture Amount
    entitlement    = parse_dollar(row[6]) if len(row) > 6 else 0.0
    recapture      = parse_dollar(row[8]) if len(row) > 8 else 0.0
    local_revenues = parse_dollar(row[5]) if len(row) > 5 else 0.0

    net_flow = entitlement - recapture

    prefix = district_id[:2]
    if prefix not in DISTRICT_PREFIX_TO_GEOID:
        print(f"  WARNING: unknown prefix '{prefix}' for district {district_id} ({name})")
        continue
    geoid = DISTRICT_PREFIX_TO_GEOID[prefix]

    district_flows[district_id] = {
        "name":           name,
        "district_id":    district_id,
        "prefix":         prefix,
        "geoid":          geoid,
        "entitlement":    entitlement,
        "recapture":      recapture,
        "local_revenues": local_revenues,
        "net_flow":       net_flow,
    }

print(f"\nParsed {len(district_flows)} districts from Payment Details CSV")

# ── DQ Check: no district should have both entitlement AND recapture > 0.01 ───
both_nonzero = [
    d for d in district_flows.values()
    if d["entitlement"] > 0.01 and d["recapture"] > 0.01
]
if both_nonzero:
    print(f"\n⚠ DQ WARNING: {len(both_nonzero)} district(s) have both entitlement AND recapture > 0:")
    for d in both_nonzero:
        print(f"   {d['district_id']} {d['name']}: entitlement=${d['entitlement']:,.2f}, recapture=${d['recapture']:,.2f}")
else:
    print("\n✓ DQ: no district has both entitlement and recapture nonzero")

# ── Recapture districts table ──────────────────────────────────────────────────
recap_districts = [d for d in district_flows.values() if d["recapture"] > 0]
if recap_districts:
    print(f"\nRecapture districts ({len(recap_districts)}):")
    print(f"  {'District ID':<12} {'Name':<20} {'County':<14} {'Recapture':>16} {'Net Flow':>16}")
    print("  " + "-" * 82)
    for d in sorted(recap_districts, key=lambda x: x["district_id"]):
        county = GEOID_TO_COUNTY[d["geoid"]]
        print(f"  {d['district_id']:<12} {d['name']:<20} {county:<14} "
              f"${d['recapture']:>15,.2f} ${d['net_flow']:>15,.2f}")
else:
    print("No recapture districts found.")


In [ ]:
# Cell 4: Parse LCR for 6-mill weights and mineral revenue

def parse_dollar(s):
    if s is None: return 0.0
    s = str(s).strip().replace(',', '').replace('$', '').replace(' ', '')
    if not s or s == '-': return 0.0
    s = s.replace('(', '-').replace(')', '')
    try: return float(s)
    except: return 0.0

with open(LCR_CSV) as f:
    reader = csv.reader(f)
    lcr_rows = list(reader)

# Verify column headers
h1, h2 = lcr_rows[0], lcr_rows[1]
print('LCR header col 34:', h1[34])
print('LCR header col 35:', h1[35])
print('LCR header col 50:', h1[50])
print('LCR header col 51:', h1[51][:50])  # a) deferred mineral 25-mill
print('LCR header col 52:', h1[52][:50])  # b) deferred mineral 6-mill
print('LCR header col 55:', h1[55][:50])  # e) PY2024 25-mill mineral
print('LCR header col 56:', h1[56][:50])  # f) PY2024 6-mill mineral
print('LCR header col 57:', h1[57][:50])  # g) PY2025 25-mill mineral
print('LCR header col 58:', h1[58][:50])  # h) PY2025 6-mill mineral
print('LCR header col 59:', h1[59][:60])  # Total Estimated FY2025
print()

lcr_data = {}
for row in lcr_rows[2:]:
    if not row or not row[0].strip(): continue
    raw_id = row[0].strip()
    if raw_id.startswith('77'): continue
    dist_id = raw_id.zfill(7)
    mill6_own   = parse_dollar(row[34] if len(row) > 34 else '')
    mill6_cross = parse_dollar(row[35] if len(row) > 35 else '')
    mill6_total = mill6_own + mill6_cross
    local_rev   = parse_dollar(row[50] if len(row) > 50 else '')
    # Mineral columns: all periods a+b+e+f+g+h
    min_a = parse_dollar(row[51] if len(row) > 51 else '')  # deferred 25-mill mineral
    min_b = parse_dollar(row[52] if len(row) > 52 else '')  # deferred 6-mill mineral
    min_e = parse_dollar(row[55] if len(row) > 55 else '')  # PY2024 25-mill mineral
    min_f = parse_dollar(row[56] if len(row) > 56 else '')  # PY2024 6-mill mineral
    min_g = parse_dollar(row[57] if len(row) > 57 else '')  # PY2025 25-mill mineral
    min_h = parse_dollar(row[58] if len(row) > 58 else '')  # PY2025 6-mill mineral
    total_mineral = min_a + min_b + min_e + min_f + min_g + min_h
    total_est_fy2025 = parse_dollar(row[59] if len(row) > 59 else '')  # denominator
    lcr_data[dist_id] = {
        'mill6_own': mill6_own, 'mill6_cross': mill6_cross, 'mill6_total': mill6_total,
        'local_rev': local_rev,
        'total_mineral': total_mineral, 'total_est_fy2025': total_est_fy2025,
    }

print(f'Parsed {len(lcr_data)} districts from LCR CSV')
print()
print(f'Cross-county districts (mill6_cross > 0):')
print(f'  {"District ID":<12} {"Name":<26} {"82110 (own)":>14} {"82111 (cross)":>14}   Home%  Cross%')
print('  ' + '-'*90)
for d_id in sorted(lcr_data):
    d = lcr_data[d_id]
    if d['mill6_cross'] > 0:
        hp = d['mill6_own']/d['mill6_total']*100 if d['mill6_total'] else 0
        name = next((v for k,v in district_flows.items() if k == d_id), {}).get('name', '')
        print(f'  {d_id:<12} {name:<26} {d["mill6_own"]:>14,.2f} {d["mill6_cross"]:>14,.2f}   {hp:4.1f}%  {100-hp:4.1f}%')


In [ ]:
# Cell 4: Parse LCR for 6-mill weights and mineral revenue

def parse_dollar(s):
    if s is None: return 0.0
    s = str(s).strip().replace(',', '').replace('$', '').replace(' ', '')
    if not s or s == '-': return 0.0
    s = s.replace('(', '-').replace(')', '')
    try: return float(s)
    except: return 0.0

with open(LCR_CSV) as f:
    reader = csv.reader(f)
    lcr_rows = list(reader)

# Verify column headers
h1, h2 = lcr_rows[0], lcr_rows[1]
print('LCR header col 34:', h1[34])
print('LCR header col 35:', h1[35])
print('LCR header col 50:', h1[50])
print('LCR header col 51:', h1[51][:50])  # a) deferred mineral 25-mill
print('LCR header col 52:', h1[52][:50])  # b) deferred mineral 6-mill
print('LCR header col 55:', h1[55][:50])  # e) PY2024 25-mill mineral
print('LCR header col 56:', h1[56][:50])  # f) PY2024 6-mill mineral
print('LCR header col 57:', h1[57][:50])  # g) PY2025 25-mill mineral
print('LCR header col 58:', h1[58][:50])  # h) PY2025 6-mill mineral
print('LCR header col 59:', h1[59][:60])  # Total Estimated FY2025
print()

lcr_data = {}
for row in lcr_rows[2:]:
    if not row or not row[0].strip(): continue
    raw_id = row[0].strip()
    if raw_id.startswith('77'): continue
    dist_id = raw_id.zfill(7)
    mill6_own   = parse_dollar(row[34] if len(row) > 34 else '')
    mill6_cross = parse_dollar(row[35] if len(row) > 35 else '')
    mill6_total = mill6_own + mill6_cross
    local_rev   = parse_dollar(row[50] if len(row) > 50 else '')
    # Mineral columns: all periods a+b+e+f+g+h
    min_a = parse_dollar(row[51] if len(row) > 51 else '')  # deferred 25-mill mineral
    min_b = parse_dollar(row[52] if len(row) > 52 else '')  # deferred 6-mill mineral
    min_e = parse_dollar(row[55] if len(row) > 55 else '')  # PY2024 25-mill mineral
    min_f = parse_dollar(row[56] if len(row) > 56 else '')  # PY2024 6-mill mineral
    min_g = parse_dollar(row[57] if len(row) > 57 else '')  # PY2025 25-mill mineral
    min_h = parse_dollar(row[58] if len(row) > 58 else '')  # PY2025 6-mill mineral
    total_mineral = min_a + min_b + min_e + min_f + min_g + min_h
    total_est_fy2025 = parse_dollar(row[59] if len(row) > 59 else '')  # denominator
    lcr_data[dist_id] = {
        'mill6_own': mill6_own, 'mill6_cross': mill6_cross, 'mill6_total': mill6_total,
        'local_rev': local_rev,
        'total_mineral': total_mineral, 'total_est_fy2025': total_est_fy2025,
    }

print(f'Parsed {len(lcr_data)} districts from LCR CSV')
print()
print(f'Cross-county districts (mill6_cross > 0):')
print(f'  {"District ID":<12} {"Name":<26} {"82110 (own)":>14} {"82111 (cross)":>14}   Home%  Cross%')
print('  ' + '-'*90)
for d_id in sorted(lcr_data):
    d = lcr_data[d_id]
    if d['mill6_cross'] > 0:
        hp = d['mill6_own']/d['mill6_total']*100 if d['mill6_total'] else 0
        name = next((v for k,v in district_flows.items() if k == d_id), {}).get('name', '')
        print(f'  {d_id:<12} {name:<26} {d["mill6_own"]:>14,.2f} {d["mill6_cross"]:>14,.2f}   {hp:4.1f}%  {100-hp:4.1f}%')


In [ ]:
# ── Zero leakage assertion ─────────────────────────────────────────────────────
total_district = sum(d["net_flow"] for d in district_flows.values())
total_county   = sum(county_net_flow.values())

assert abs(total_district - total_county) < 1.0, (
    f"LEAKAGE: district total = {total_district:.2f}, county total = {total_county:.2f}, "
    f"diff = {abs(total_district - total_county):.4f}"
)

print(f"✓ Zero-leakage check passed")
print(f"  District total net_flow: ${total_district:,.2f}")
print(f"  County total net_flow:   ${total_county:,.2f}")
print(f"  Difference:              ${abs(total_district - total_county):.4f}")


In [ ]:
# Cell 7: Mineral-attributable share (school_finance_mineral_share)
# 
# Numerator: total mineral across ALL periods (a+b+e+f+g+h)
# Denominator: Total Estimated FY2025 25&6-mill Tax Collections (col 59)
# 
# Rationale: Some counties (Campbell, Converse) show PY2024 mineral=0 with
# PY2025 holding the mineral estimates due to billing/assessment timing.
# Using total across all periods gives the correct mineral fraction.

# Precompute per-county 82110 totals for 3-county cluster weighting
cluster_county_mill6own = defaultdict(float)
for d_id, d in lcr_data.items():
    prefix = d_id[:2]
    if prefix in DISTRICT_PREFIX_TO_GEOID:
        cluster_county_mill6own[DISTRICT_PREFIX_TO_GEOID[prefix]] += d['mill6_own']

county_mineral = defaultdict(lambda: {'mineral': 0.0, 'total_est': 0.0})

for dist_id, dist in district_flows.items():
    prefix = dist_id[:2]
    home_geoid = dist['geoid']
    lcr = lcr_data.get(dist_id)
    if not lcr:
        continue
    total_mineral  = lcr['total_mineral']
    total_est      = lcr['total_est_fy2025']
    mill6_own      = lcr['mill6_own']
    mill6_cross    = lcr['mill6_cross']
    mill6_total    = lcr['mill6_total']

    cross_info = None
    for cluster_map in CROSS_COUNTY_CLUSTERS.values():
        if prefix in cluster_map:
            cross_info = cluster_map[prefix]
            break

    if cross_info is None or mill6_total == 0:
        county_mineral[home_geoid]['mineral']   += total_mineral
        county_mineral[home_geoid]['total_est'] += total_est
    else:
        home_geoid_check, cross_dest = cross_info
        home_share  = mill6_own / mill6_total if mill6_total > 0 else 1.0
        cross_share = 1.0 - home_share
        county_mineral[home_geoid]['mineral']   += home_share * total_mineral
        county_mineral[home_geoid]['total_est'] += home_share * total_est
        if cross_dest is None:
            pass
        elif isinstance(cross_dest, str):
            county_mineral[cross_dest]['mineral']   += cross_share * total_mineral
            county_mineral[cross_dest]['total_est'] += cross_share * total_est
        else:
            other_mill6  = {g: cluster_county_mill6own[g] for g in cross_dest}
            total_other  = sum(other_mill6.values())
            for og in cross_dest:
                w = other_mill6[og] / total_other if total_other > 0 else 0.5
                county_mineral[og]['mineral']   += cross_share * w * total_mineral
                county_mineral[og]['total_est'] += cross_share * w * total_est

# Compute mineral share per county
county_mineral_share = {}
for geoid in DISTRICT_PREFIX_TO_GEOID.values():
    county_mineral.setdefault(geoid, {'mineral': 0.0, 'total_est': 0.0})
    m = county_mineral[geoid]
    pct = m['mineral'] / m['total_est'] if m['total_est'] > 0 else 0.0
    pct = min(pct, 1.0)  # cap at 100%
    county_mineral_share[geoid] = round(pct, 4)

print(f'{"County":<18} {"GEOID":<8} {"Mineral ($)":>16} {"TotalEst25+6":>16} {"Mineral%":>10}')
print('-'*75)
for geoid in sorted(county_mineral_share):
    m = county_mineral[geoid]
    print(f'{GEOID_TO_COUNTY[geoid]:<18} {geoid:<8} {m["mineral"]:>14,.0f} {m["total_est"]:>14,.0f} {county_mineral_share[geoid]*100:>9.1f}%')


## Vintage Mismatch Documentation

Three distinct vintages are used in this notebook and the broader `wy_fiscal_coefficients.json` file. They are **not forced to reconcile**.

### 1. Payment Details: FY2025 WDE Funding Model Version 3(d)
- Vintage: FY2025 model output (estimated payments for the FY2025 school year)
- Used for: `school_finance_net_total` = entitlement_amount − recapture_amount
- Source: *Wyoming Block Grant Estimated Payment Details  - Sheet1.csv*, dated 6/17/2026

### 2. Locally Collected Revenues: FY2024 actuals + PY2024/2025 estimates
- Vintage (6-mill weights): FY2024 actual collections (columns 82110/82111)
- Vintage (mineral share, denominator): Total Estimated FY2025 25&6-mill Tax Collections (col 59)
- Vintage (mineral share, numerator): All mineral-attributed columns (a+b+e+f+g+h = deferred + PY2024 + PY2025)
- **Why total across all periods?** Some counties (Campbell, Converse, Teton) show PY2024 mineral = $0 with PY2025 holding the large mineral estimates — likely due to billing/assessment cycle timing where coal/oil assessment occurs on a different schedule. Using the sum of all periods (a+b+e+f+g+h) divided by col[59] (total estimated FY2025 collections) gives the correct mineral fraction for each county. For Campbell: 78.2% mineral; Converse: 85.6%; Teton: 0.0% (resort county, no mineral extraction).
- Source: *Wyoming School District Locally Collected Revenues  - Sheet1.csv*

### 3. DOR Assessed Values: WY DOR 2025 Annual Report
- Used in `property_tax_annual`, `valuation_delta`, and other fields in the same JSON file
- A distinct report year from both of the above school finance sources
- Not reconciled to school finance data here

### Non-reconciliation rationale
The FY2025 model entitlement/recapture is compared to FY2024 actual local revenues in some aggregations. This is acceptable because the foundation program guarantee is computed from a prior-year base — the same approach used when documenting the Lincoln Naughton −92.8% back-cast residual in NB18 (W2). Vintage mismatches are documented, not smoothed.


In [ ]:
# ── Before / After comparison table ───────────────────────────────────────────
OLD_PROXY = 6084650.0  # constant from 37-mill proxy for wind_utility (same for all counties)

print(f"{'County':<18} {'GEOID':<8} {'Old 37-mill':>14} {'New WDE FY25':>14} {'% Diff':>10}  Flag")
print("-" * 75)

sign_flips   = []
large_shifts = []

for geoid in sorted(county_net_flow):
    name    = GEOID_TO_COUNTY[geoid]
    new_val = county_net_flow[geoid]
    pct_diff = (new_val - OLD_PROXY) / abs(OLD_PROXY) * 100
    flag = ""
    if (OLD_PROXY > 0) != (new_val > 0):
        flag = "\u26a0 SIGN FLIP"
        sign_flips.append(geoid)
    elif abs(pct_diff) > 100:
        flag = "\u26a0 >100% SHIFT"
        large_shifts.append(geoid)
    elif abs(pct_diff) > 50:
        flag = "\u25b3 >50% SHIFT"
    print(f"{name:<18} {geoid:<8} {OLD_PROXY:>14,.0f} {new_val:>14,.0f} {pct_diff:>+9.1f}%  {flag}")

print()
if sign_flips:
    print(f"Sign-flip counties ({len(sign_flips)}): {[GEOID_TO_COUNTY[g] for g in sign_flips]}")
if large_shifts:
    print(f">100%-shift counties ({len(large_shifts)}): {[GEOID_TO_COUNTY[g] for g in large_shifts]}")

campbell_new = county_net_flow["56005"]
laramie_new  = county_net_flow["56021"]
print()
print(f"=== W3 Golden D Relevance ===")
print(f"Campbell (56005): OLD=${OLD_PROXY:,.0f}  NEW=${campbell_new:,.0f}  diff=${campbell_new - OLD_PROXY:+,.0f}")
if campbell_new < 0:
    print(f"  → Campbell is a RECAPTURE county. Net_flow < 0 means Campbell districts")
    print(f"    recapture more than they receive from the foundation guarantee.")
    print(f"  → In a coal retirement scenario, as mineral value drops, recapture DECREASES")
    print(f"    (becomes less negative = fiscal gain to the district).")
    print(f"  → W3 must model this sign correctly in the fiscal arc.")
print(f"Laramie (56021): OLD=${OLD_PROXY:,.0f}  NEW=${laramie_new:,.0f}  diff=${laramie_new - OLD_PROXY:+,.0f}")


In [ ]:
# ── Update wy_fiscal_coefficients.json ────────────────────────────────────────
with open(COEFF_FILE) as f:
    coeffs = json.load(f)

# Build new school_finance_net structure (county-level, action-independent)
new_sfn = {}
for geoid in DISTRICT_PREFIX_TO_GEOID.values():
    new_sfn[geoid] = {
        "school_finance_net_total": {
            "value":      round(county_net_flow[geoid], 2),
            "unit":       "USD/yr",
            "source":     "Wyoming Block Grant Estimated Payment Details, FY2025 WDE Funding Model Version 3(d)",
            "vintage":    "FY2025",
            "confidence": "medium",
            "notes": (
                "net_flow = entitlement_amount - recapture_amount, aggregated from district to county "
                "via 6-mill (82110/82111) cross-county weights. Four cross-county clusters: "
                "Big Horn/Park, Lincoln/Sublette, Goshen/Niobrara/Platte, Carbon/Sweetwater. "
                "Negative = county districts are net recapture contributors to state. "
                "Supersedes 37-mill proxy (confidence was: low)."
            ),
        },
        "school_finance_mineral_share": {
            "value":      county_mineral_share[geoid]["mineral_share_pct"],
            "unit":       "fraction",
            "source":     "Wyoming School District Locally Collected Revenues, WDE",
            "vintage":    "PY2024 mineral (cols e+f); FY2024 actuals (denominator)",
            "confidence": "medium",
            "notes": (
                "Numerator: PY2024 25-mill + 6-mill taxes attributable to minerals. "
                "Denominator: FY2024 revenue counted as local revenue. "
                "Vintage mismatch documented in NB18b. "
                "Stored as informational field — not blended into net_flow."
            ),
        },
    }

# Replace school_finance_net in all 49 actions
patched_count = 0
for action_id in coeffs:
    if "school_finance_net" in coeffs[action_id]:
        coeffs[action_id]["school_finance_net"] = new_sfn
        patched_count += 1

with open(COEFF_FILE, "w") as f:
    json.dump(coeffs, f, indent=2)

print(f"✓ Written {COEFF_FILE}")
print(f"  Patched {patched_count} actions")
print()

# Verify round-trip
with open(COEFF_FILE) as f:
    v = json.load(f)
action0 = list(v.keys())[0]
sfn_campbell = v[action0]["school_finance_net"]["56005"]
print(f"Verification — Campbell (56005) school_finance_net [{action0}]:")
print(f"  school_finance_net_total.value        = ${sfn_campbell['school_finance_net_total']['value']:,.2f}")
print(f"  school_finance_net_total.vintage      = {sfn_campbell['school_finance_net_total']['vintage']}")
print(f"  school_finance_net_total.confidence   = {sfn_campbell['school_finance_net_total']['confidence']}")
print(f"  school_finance_mineral_share.value    = {sfn_campbell['school_finance_mineral_share']['value']:.4f}")
print(f"  school_finance_mineral_share.vintage  = {sfn_campbell['school_finance_mineral_share']['vintage']}")


In [ ]:
# ── Update wy_fiscal_sources.csv ──────────────────────────────────────────────
new_rows = [
    [
        "statewide",
        "school_finance_net_total",
        "entitlement_amount - recapture_amount (district\u2192county aggregated)",
        "USD/yr",
        "all 23 WY counties / 48 school districts",
        "school_finance_net",
        "Wyoming Block Grant Estimated Payment Details FY2025 WDE Funding Model Version 3(d)",
        2025,
        (
            "Cross-county apportionment via 82110/82111 6-mill split. "
            "Four clusters: Big Horn/Park, Lincoln/Sublette, Goshen/Niobrara/Platte, Carbon/Sweetwater. "
            "Replaces 37-mill proxy. "
            "Paired source: District Locally Collected Revenues (FY2024/PY2024)."
        ),
        "medium",
    ],
    [
        "statewide",
        "school_finance_mineral_share",
        "PY2024_mineral_25mill_plus_6mill / FY2024_local_revenue_counted",
        "fraction",
        "all 23 WY counties / 48 school districts",
        "school_finance_net",
        "Wyoming School District Locally Collected Revenues FY2024/PY2024 (WDE)",
        2024,
        (
            "Numerator: PY2024 mineral estimates (cols e+f). "
            "Denominator: FY2024 actuals. "
            "Vintage mismatch: FY2025 model vs FY2024/PY2024 revenues — not forced to reconcile. "
            "Informational field only."
        ),
        "medium",
    ],
]

with open(SOURCES_FILE, "a", newline="") as f:
    writer = csv.writer(f)
    for row in new_rows:
        writer.writerow(row)

print(f"\u2713 Appended 2 rows to {SOURCES_FILE}")

# Verify
with open(SOURCES_FILE, newline='', encoding='utf-8') as f:
    all_source_rows = list(csv.reader(f))
print(f"  Total rows now: {len(all_source_rows)} (was {len(all_source_rows)-2})")
print(f"  Last 2 rows:")
for row in all_source_rows[-2:]:
    print(f"    {row[:4]}...")


## Handoff Conditions — NB18b Complete

- [ ] All 23 counties have `school_finance_net_total` from FY2025 WDE model (not 37-mill proxy)
- [ ] Cross-county apportionment: 4 clusters, zero-leakage assertion passed
- [ ] DQ: no district with both entitlement > 0 and recapture > 0
- [ ] `school_finance_mineral_share` added (PY2024, separately documented)
- [ ] Three vintage mismatches documented (Cell 8)
- [ ] Before/after table with Campbell & Laramie W3 flags
- [ ] `wy_fiscal_sources.csv` updated (+2 medium-confidence rows)
- [ ] Old fields removed: `local_school_levy_revenue`, `foundation_recapture`, `foundation_guarantee_transfer`
